# Lesson 1.2 — Angular Velocity and the Skew Operator
**Module 6 · Unit 1 · Lesson 2**

Verifies: (1) `vee` inverts `skew`; (2) integrating $\dot R = S(\omega_s)R$ keeps $R$ a rotation and matches the closed-form spin; (3) $\omega_s = R\,\omega_b$.

Convention D-057: twist $[v;\omega]$, base/world frame.

In [1]:
import numpy as np
checks=[]
def skew(v):
    v=np.asarray(v,float).ravel()
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S):
    # inverse of skew: pull the axis-vector out of a skew matrix
    return np.array([S[2,1],S[0,2],S[1,0]])
w=np.array([0.3,-1.2,0.7])
checks.append(np.allclose(vee(skew(w)), w))
print("vee(skew(w)) == w :", checks[-1])

vee(skew(w)) == w : True


## Integrate $\dot R = S(\omega_s)R$ and confirm $R$ stays a rotation

In [2]:
def R_exact(k,ang):
    k=np.asarray(k,float); k=k/np.linalg.norm(k); K=skew(k)
    return np.eye(3)+np.sin(ang)*K+(1-np.cos(ang))*(K@K)

omega_s=np.array([0,0,1.0])       # constant spin about world z, 1 rad/s
R=np.eye(3); dt=1e-4
for _ in range(int(1.0/dt)):
    R=R+dt*(skew(omega_s)@R)       # Euler step of Rdot = S(w)R
    U,_,Vt=np.linalg.svd(R); R=U@Vt # re-orthonormalize (remove integration drift)

ortho_err=np.linalg.norm(R.T@R-np.eye(3))
det_err=abs(np.linalg.det(R)-1)
match_err=np.linalg.norm(R - R_exact([0,0,1],1.0))   # should equal a 1 rad spin about z
print(f"orthonormality error = {ortho_err:.2e}")
print(f"det(R)-1            = {det_err:.2e}")
print(f"vs closed-form spin = {match_err:.2e}")
checks += [ortho_err<1e-9, det_err<1e-9, match_err<1e-3]

orthonormality error = 3.55e-16
det(R)-1            = 2.22e-16
vs closed-form spin = 4.71e-09


## Spatial vs body angular velocity:  $\omega_s = R\,\omega_b$

In [3]:
R=R_exact([0.4,0.5,0.77],0.9)
omega_b=np.array([0.2,-0.5,0.9])
omega_s=R@omega_b
# Rdot built two ways must agree: S(w_s)R  ==  R S(w_b)
checks.append(np.allclose(skew(omega_s)@R, R@skew(omega_b)))
print("S(w_s)R == R S(w_b) :", checks[-1])
print("recovered w_s via (Rdot Rt)^vee :", vee((skew(omega_s)@R)@R.T))

S(w_s)R == R S(w_b) : True
recovered w_s via (Rdot Rt)^vee : [ 0.8563839  -0.37327398  0.47673173]


In [4]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
